<a href="https://colab.research.google.com/github/rezamarzban/fidelio/blob/master/fidelio_pico_build.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Fidelio (FIDO2/U2F key) for Raspberry Pi Pico
Run the cells top to bottom. Set `BOARD` below (`pico` or `rp2040-zero`).

In [1]:
# 1. Install build tools (CMake >= 3.31 via pip)
!apt-get -qq update
!apt-get -qq install -y gcc-arm-none-eabi libnewlib-arm-none-eabi libstdc++-arm-none-eabi-newlib build-essential xxd openssl git
!pip -q install "cmake>=3.31"
!cmake --version | head -1
!arm-none-eabi-gcc --version | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../libssl-dev_3.0.13-0ubuntu3.16_amd64.deb ...
Unpacking libssl-dev:amd64 (3.0.13-0ubuntu3.16) over (3.0.13-0ubuntu3.15) ...
Preparing to unpack .../libssl3t64_3.0.13-0ubuntu3.16_amd64.deb ...
Unpacking libssl3t64:amd64 (3.0.13-0ubuntu3.16) over (3.0.13-0ubuntu3.15) ...
Setting up libssl3t64:amd64 (3.0.13-0ubuntu3.16) ...
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../0-openssl_3.0.13-0ubuntu3.16_amd64.deb ...
Unpacking openssl (3.0.13-0ubuntu3.16) over (3.0.13-0ubuntu3.15) ...
Selecting previously unselected package xxd.
Preparing to unpack .../1-xxd_2%3a9.1.0016-1ubuntu7.20_amd64.deb ...
Unpacking xxd (2:9.1.0016-1ubuntu7.20) ...
Selecting previously unsele

In [2]:
# 2. Clone repo and submodules
%cd /content
!rm -rf fidelio
!git clone https://github.com/danielinux/fidelio.git
%cd /content/fidelio
!git submodule update --init --single-branch pico-sdk lib/wolfssl lib/wolfcose
!git -C pico-sdk submodule update --init --single-branch lib/tinyusb

/content
Cloning into 'fidelio'...
remote: Enumerating objects: 361, done.
remote: Counting objects: 100% (281/281), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 361 (delta 176), reused 246 (delta 151), pack-reused 80 (from 1)
Receiving objects: 100% (361/361), 395.67 KiB | 11.30 MiB/s, done.
Resolving deltas: 100% (209/209), done.
/content/fidelio
Submodule 'lib/wolfcose' (https://github.com/wolfSSL/wolfCOSE) registered for path 'lib/wolfcose'
Submodule 'lib/wolfssl' (https://github.com/wolfssl/wolfssl) registered for path 'lib/wolfssl'
Submodule 'pico-sdk' (https://github.com/raspberrypi/pico-sdk) registered for path 'pico-sdk'
Cloning into '/content/fidelio/lib/wolfcose'...
Cloning into '/content/fidelio/lib/wolfssl'...
Cloning into '/content/fidelio/pico-sdk'...
Submodule path 'lib/wolfcose': checked out 'f907071b10127f3ae2dd7719749a91b039ff04a1'
Submodule path 'lib/wolfssl': checked out '14a72c96e4fab27b5bfafc606fdd24b52338842e'
Submodule path 'pico-sdk':

In [3]:

from google.colab import drive
drive.mount('/content/drive')

!tar -czf /content/drive/MyDrive/fidelio_backup.tar.gz -C /content fidelio

print("Backup completed.")

Mounted at /content/drive
Backup completed.


In [4]:
# 3. Create attestation certificate (embeds it in src/cert.c)
# Optional: edit attestation-cert.conf first.
# !./mkcert.sh

In [5]:
# 4. Build
import os
os.environ['BOARD'] = 'pico'   # or 'rp2040-zero'
!cmake -S . -B build -DFAMILY=rp2040 -DPICO_SDK_PATH="$PWD/pico-sdk" -DBOARD=$BOARD
!cmake --build build -j$(nproc)
!ls -l build/fidelio.uf2

-- Fidelio target board: pico (FIDELIO_BOARD_PICO)
PICO_SDK_PATH is /content/fidelio/pico-sdk
Defaulting PICO_PLATFORM to rp2040 since not specified.
Defaulting PICO platform compiler to pico_arm_gcc since not specified.
-- Defaulting build type to 'Release' since not specified.
PICO compiler is pico_arm_gcc
-- The C compiler identification is GNU 13.2.1
-- The CXX compiler identification is GNU 13.2.1
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/arm-none-eabi-gcc
Build type is Release
Defaulting PICO target board to pico since not specified.
Using board configuration from /content/fidelio/pico-sdk/src/boards/include/boards/pico.h
-- Found Python3: /usr/local/bin/python (found version "3.13.15") found components: Interpreter
TinyUSB available at /content/fidelio/pico-sdk/lib/tinyusb/src/portable/raspberrypi/rp2040; enabling build support for USB.
-- Configuring done (1.5s)
-- Generating done (0.1s)
-- Build files have been written to: /content/fidelio/build
[ 

In [6]:
# 5. Download the firmware (and the attestation key as a private backup)
from google.colab import files
files.download('build/fidelio.uf2')
# Keep this file secret; it is your attestation private key.
# files.download('cert-master-key.pem')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>